In [500]:
#TODO
#implement tiebreaks into sackmann data
# - clutch factors glickos (deciding set wins, break points converted, break points saved)
# - H2H (overall and surface)
# - recent performance stats (for model and dash)
# - create dataframe for ML training (one row per match)
# - use ML to tune glicko params

# target variable: for each match, randomly choose to flag player win (1) or opponent (0) to get even balance 


In [529]:
from data_pull import *
import pandas as pd
import numpy as np
import math
from rapidfuzz import fuzz, process
from tqdm import tqdm
from scipy import stats
import random
random_seed = 42 #TODO add to config file
random.seed(random_seed)


In [523]:




#: The actual score for win
WIN = 1.
#: The actual score for draw
DRAW = 0.5
#: The actual score for loss
LOSS = 0.


class Rating(object):
    def __init__(self, mu, phi, sigma):
        self.mu = mu
        self.phi = phi
        self.sigma = sigma

    def __repr__(self):
        c = type(self)
        args = (c.__module__, c.__name__, self.mu, self.phi, self.sigma)
        return '%s.%s(mu=%.3f, phi=%.3f, sigma=%.3f)' % args


class Glicko2(object):
    def __init__(self, mu, phi, sigma, tau, epsilon):
        self.mu = mu
        self.phi = phi
        self.sigma = sigma
        self.tau = tau
        self.epsilon = epsilon

    def create_rating(self, mu=None, phi=None, sigma=None):
        if mu is None:
            mu = self.mu
        if phi is None:
            phi = self.phi
        if sigma is None:
            sigma = self.sigma
        return Rating(mu, phi, sigma)

    def scale_down(self, rating, ratio=173.7178):
        mu = (rating.mu - self.mu) / ratio
        phi = rating.phi / ratio
        return self.create_rating(mu, phi, rating.sigma)

    def scale_up(self, rating, ratio=173.7178):
        mu = rating.mu * ratio + self.mu
        phi = rating.phi * ratio
        return self.create_rating(mu, phi, rating.sigma)

    def reduce_impact(self, rating):
        """The original form is `g(RD)`. This function reduces the impact of
        games as a function of an opponent's RD.
        """
        return 1. / math.sqrt(1 + (3 * rating.phi ** 2) / (math.pi ** 2))

    def expect_score(self, rating, other_rating, impact):
        return 1. / (1 + math.exp(-impact * (rating.mu - other_rating.mu)))

    def determine_sigma(self, rating, difference, variance):
        """Determines new sigma."""
        phi = rating.phi
        difference_squared = difference ** 2
        # 1. Let a = ln(s^2), and define f(x)
        alpha = math.log(rating.sigma ** 2)

        def f(x):
            """This function is twice the conditional log-posterior density of
            phi, and is the optimality criterion.
            """
            tmp = phi ** 2 + variance + math.exp(x)
            a = math.exp(x) * (difference_squared - tmp) / (2 * tmp ** 2)
            b = (x - alpha) / (self.tau ** 2)
            return a - b

        # 2. Set the initial values of the iterative algorithm.
        a = alpha
        if difference_squared > phi ** 2 + variance:
            b = math.log(difference_squared - phi ** 2 - variance)
        else:
            k = 1
            while f(alpha - k * math.sqrt(self.tau ** 2)) < 0:
                k += 1
            b = alpha - k * math.sqrt(self.tau ** 2)
        # 3. Let fA = f(A) and f(B) = f(B)
        f_a, f_b = f(a), f(b)
        # 4. While |B-A| > e, carry out the following steps.
        # (a) Let C = A + (A - B)fA / (fB-fA), and let fC = f(C).
        # (b) If fCfB < 0, then set A <- B and fA <- fB; otherwise, just set
        #     fA <- fA/2.
        # (c) Set B <- C and fB <- fC.
        # (d) Stop if |B-A| <= e. Repeat the above three steps otherwise.
        while abs(b - a) > self.epsilon:
            c = a + (a - b) * f_a / (f_b - f_a)
            f_c = f(c)
            if f_c * f_b < 0:
                a, f_a = b, f_b
            else:
                f_a /= 2
            b, f_b = c, f_c
        # 5. Once |B-A| <= e, set s' <- e^(A/2)
        return math.exp(1) ** (a / 2)

    def rate(self, rating, series):
        # Step 2. For each player, convert the rating and RD's onto the
        #         Glicko-2 scale.
        rating = self.scale_down(rating)
        # Step 3. Compute the quantity v. This is the estimated variance of the
        #         team's/player's rating based only on game outcomes.
        # Step 4. Compute the quantity difference, the estimated improvement in
        #         rating by comparing the pre-period rating to the performance
        #         rating based only on game outcomes.
        variance_inv = 0
        difference = 0
        if not series:
            # If the team didn't play in the series, do only Step 6
            phi_star = math.sqrt(rating.phi ** 2 + rating.sigma ** 2)
            return self.scale_up(self.create_rating(rating.mu, phi_star, rating.sigma))
        for actual_score, other_rating in series:
            other_rating = self.scale_down(other_rating)
            impact = self.reduce_impact(other_rating)
            expected_score = self.expect_score(rating, other_rating, impact)
            variance_inv += impact ** 2 * expected_score * (1 - expected_score)
            difference += impact * (actual_score - expected_score)
        difference /= variance_inv
        variance = 1. / variance_inv
        # Step 5. Determine the new value, Sigma', ot the sigma. This
        #         computation requires iteration.
        sigma = self.determine_sigma(rating, difference, variance)
        # Step 6. Update the rating deviation to the new pre-rating period
        #         value, Phi*.
        phi_star = math.sqrt(rating.phi ** 2 + sigma ** 2)
        # Step 7. Update the rating and RD to the new values, Mu' and Phi'.
        phi = 1. / math.sqrt(1 / phi_star ** 2 + 1 / variance)
        mu = rating.mu + phi ** 2 * (difference / variance)
        # Step 8. Convert ratings and RD's back to original scale.
        return self.scale_up(self.create_rating(mu, phi, sigma))

    def rate_1vs1(self, rating1, rating2, drawn=False):
        return (self.rate(rating1, [(DRAW if drawn else WIN, rating2)]),
                self.rate(rating2, [(DRAW if drawn else LOSS, rating1)]))

    def quality_1vs1(self, rating1, rating2):
        expected_score1 = self.expect_score(rating1, rating2, self.reduce_impact(rating1))
        expected_score2 = self.expect_score(rating2, rating1, self.reduce_impact(rating2))
        expected_score = (expected_score1 + expected_score2) / 2
        return 2 * (0.5 - abs(0.5 - expected_score))
    
class almost(object):

    def __init__(self, val, precision=3):
        self.val = val
        self.precision = precision

    def almost_equals(self, val1, val2):
        if round(val1, self.precision) == round(val2, self.precision):
            return True
        fmt = '%.{0}f'.format(self.precision)
        mantissa = lambda f: int((fmt % f).replace('.', ''))
        return abs(mantissa(val1) - mantissa(val2)) <= 1

    def __eq__(self, other):
        try:
            if not self.almost_equals(self.val.volatility, other.volatility):
                return False
        except AttributeError:
            pass
        return (self.almost_equals(self.val.mu, other.mu) and
                self.almost_equals(self.val.sigma, other.sigma))

    def __repr__(self):
        return repr(self.val)
    



In [524]:
match_stats = pd.read_csv("../data/prep/match_stats.csv")
match_stats.drop([c for c in match_stats.columns if "Unnamed" in c or "index" in c or "level_0" in c],axis=1,inplace=True)

match_stats_atp = match_stats[match_stats.competition_category == "ATP"].sort_values(["match_date","event_id"])
match_stats_atp = match_stats_atp.reset_index()
unique_players = match_stats_atp.player_name.unique()

/tmp/ipykernel_10604/4131256117.py:1: DtypeWarning: Columns (4,7,8,10,11,76) have mixed types. Specify dtype option on import or set low_memory=False.
  match_stats = pd.read_csv("../data/prep/match_stats.csv")


In [525]:
glicko_dict_inputs = {"match_wins_overall":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "match_wins_clay":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "match_wins_grass":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "match_wins_hard":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "set_wins_overall":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "set_wins_clay":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "set_wins_grass":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "set_wins_hard":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                    },
                    "service_hold_wins_overall":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "return_break_wins_overall":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "service_hold_wins_clay":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "return_break_wins_clay":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "service_hold_wins_grass":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "return_break_wins_grass":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "service_hold_wins_hard":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },
                    "return_break_wins_hard":{
                        "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.01,
                            "epsilon":0.000001
                        }
                    },

                    "second_serve_win_overall":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "second_serve_return_win_overall":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },

                   "second_serve_win_clay":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "second_serve_return_win_clay":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },

                   "second_serve_win_grass":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "second_serve_return_win_grass":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },

                   "second_serve_win_hard":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "second_serve_return_win_hard":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },

                   "aces_df_diff":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "first_serve_win":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "first_serve_return_win":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   
                   "bp_conversions":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "tiebreak_overall":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "tiebreak_clay":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "tiebreak_grass":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "tiebreak_hard":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "aces":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   },
                   "aces_allowed":{
                       "init":{
                            "tau":0.5,
                            "mu":1500,
                            "phi":350,
                            "sigma": 0.06,
                            "epsilon":0.000001
                        }
                   }
    }



In [526]:
match_stats_atp["tiebreaks_won"] = match_stats_atp["tiebreaks_won"].fillna(0)
match_stats_atp["tiebreaks_lost"] = match_stats_atp["tiebreaks_lost"].fillna(0)

In [528]:
#precompute perc CDF's 
second_serve_win_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.second_serve_win_perc.notnull(),"second_serve_win_perc"]))
second_serve_return_win_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.second_serve_return_win_perc.notnull(),"second_serve_return_win_perc"]))
double_faults_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.double_faults_perc.notnull(),"double_faults_perc"]))
aces_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.aces_perc.notnull(),"aces_perc"]))
aces_allowed_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.aces_allowed_perc.notnull(),"aces_allowed_perc"]))
service_game_holds_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.service_game_holds_perc.notnull(),"service_game_holds_perc"]))
return_games_broke_perc_cdf = stats.ecdf(list(match_stats_atp.loc[match_stats_atp.return_games_broke_perc.notnull(),"return_games_broke_perc"]))

match_stats_atp["second_serve_win_perc_cdf"] = match_stats_atp["second_serve_win_perc"].apply(lambda x : second_serve_win_perc_cdf.cdf.evaluate(x))
match_stats_atp["second_serve_return_win_perc_cdf"] = match_stats_atp["second_serve_return_win_perc"].apply(lambda x : second_serve_return_win_perc_cdf.cdf.evaluate(x))
match_stats_atp["double_faults_perc_cdf"] = match_stats_atp["double_faults_perc"].apply(lambda x : double_faults_perc_cdf.cdf.evaluate(x))
match_stats_atp["aces_perc_cdf"] = match_stats_atp["aces_perc"].apply(lambda x : aces_perc_cdf.cdf.evaluate(x))
match_stats_atp["aces_allowed_perc_cdf"] = match_stats_atp["aces_allowed_perc"].apply(lambda x : aces_allowed_perc_cdf.cdf.evaluate(x))
match_stats_atp["service_game_holds_perc_cdf"] = match_stats_atp["service_game_holds_perc"].apply(lambda x : service_game_holds_perc_cdf.cdf.evaluate(x))
match_stats_atp["return_games_broke_perc_cdf"] = match_stats_atp["return_games_broke_perc"].apply(lambda x : return_games_broke_perc_cdf.cdf.evaluate(x))
#TODO: repeat for other percentages (namely breaks points converted vs saved)

In [531]:
#this dict for glicko dashboard 
atp_glicko_output_dict = {"category":[],
                    "date":[],
                    "event_id":[],
                    "player_name":[],
                    "glicko2_mu":[],
                    "glicko2_phi":[],
                    "glicko2_sigma":[]
                    }

#initialize all of the players and ratings
glicko_dict = {"envs":{}}
for category in glicko_dict_inputs.keys():
    env =  Glicko2(tau=glicko_dict_inputs[category]["init"]["tau"],
                   mu=glicko_dict_inputs[category]["init"]["mu"],
                   phi=glicko_dict_inputs[category]["init"]["phi"],
                   sigma=glicko_dict_inputs[category]["init"]["sigma"],
                   epsilon=glicko_dict_inputs[category]["init"]["epsilon"],
    )
    glicko_dict["envs"][category] = env
    glicko_dict[category] = {}
    for player in unique_players:
        glicko_dict[category][player] = env.create_rating()





#loop through match results (skip every other row for glicko purposes)
for i in tqdm(match_stats_atp[::2].index):

        player = match_stats_atp.loc[i,"player_name"]
        opponent = match_stats_atp.loc[i,"opponent_name"]
        event_id = match_stats_atp.loc[i,"event_id"]

        #match_wins

        #save most recent scores of players prior to match, use those throughout match to represent their skill level that moment
        glicko_player_prior = glicko_dict["match_wins_overall"][player]
        glicko_opp_prior = glicko_dict["match_wins_overall"][opponent]
        glicko_dict["match_wins_overall"][player] = glicko_dict["envs"]["match_wins_overall"].rate(glicko_player_prior, [(match_stats_atp.loc[i,"winner_flag"], glicko_opp_prior )])
        atp_glicko_output_dict["category"].append("match_wins_overall")
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["match_wins_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["match_wins_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["match_wins_overall"][player].sigma)

        glicko_dict["match_wins_overall"][opponent] = glicko_dict["envs"]["match_wins_overall"].rate(glicko_opp_prior, [(int(not match_stats_atp.loc[i,"winner_flag"]), glicko_player_prior )])
        atp_glicko_output_dict["category"].append("match_wins_overall")
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["match_wins_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["match_wins_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["match_wins_overall"][opponent].sigma)


        #surface match overall
        surface = match_stats_atp.loc[i,"surface"]
        if "hard" in surface.lower():
            surface = "hard"
        if "clay" in surface.lower():
            surface = "clay"

            
        if surface in ["clay","hard","grass","carpet"]:

            #save most recent scores of players prior to match, use those throughout match to represent their skill level that moment
            glicko_player_prior = glicko_dict[f"match_wins_{surface}"][player]
            glicko_opp_prior = glicko_dict[f"match_wins_{surface}"][opponent]
            glicko_dict[f"match_wins_{surface}"][player] = glicko_dict["envs"][f"match_wins_{surface}"].rate(glicko_player_prior, [(match_stats_atp.loc[i,"winner_flag"], glicko_opp_prior )])
            atp_glicko_output_dict["category"].append(f"match_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"match_wins_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"match_wins_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"match_wins_{surface}"][player].sigma)

            glicko_dict[f"match_wins_{surface}"][opponent] = glicko_dict["envs"][f"match_wins_{surface}"].rate(glicko_opp_prior, [(int(not match_stats_atp.loc[i,"winner_flag"]), glicko_player_prior )])
            atp_glicko_output_dict["category"].append(f"match_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"match_wins_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"match_wins_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"match_wins_{surface}"][opponent].sigma)


        #tiebreak_wins
        glicko_player_prior = glicko_dict[f"tiebreak_overall"][player]
        glicko_opp_prior = glicko_dict[f"tiebreak_overall"][opponent]
        player_tiebreaks_won = int(match_stats_atp["tiebreaks_won"].fillna(0).loc[i])
        opponent_tiebreaks_won = int(match_stats_atp["tiebreaks_lost"].fillna(0).loc[i])
        for twp in range(player_tiebreaks_won):
             
            glicko_dict["tiebreak_overall"][player] = glicko_dict["envs"]["tiebreak_overall"].rate(glicko_player_prior, [(WIN, glicko_opp_prior )])
            glicko_dict["tiebreak_overall"][opponent] = glicko_dict["envs"]["tiebreak_overall"].rate(glicko_opp_prior, [(LOSS, glicko_player_prior )])

        for two in range(opponent_tiebreaks_won):
             
            glicko_dict["tiebreak_overall"][player] = glicko_dict["envs"]["tiebreak_overall"].rate(glicko_player_prior, [(LOSS, glicko_opp_prior )])
            glicko_dict["tiebreak_overall"][opponent] = glicko_dict["envs"]["tiebreak_overall"].rate(glicko_opp_prior, [(WIN, glicko_player_prior )])

        atp_glicko_output_dict["category"].append("tiebreak_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["tiebreak_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["tiebreak_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["tiebreak_overall"][player].sigma)
        atp_glicko_output_dict["category"].append("tiebreak_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["tiebreak_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["tiebreak_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["tiebreak_overall"][opponent].sigma)

        #tiebreak wins surface
        if surface in ["clay","hard","grass","carpet"]:

            glicko_player_prior = glicko_dict[f"tiebreak_{surface}"][player]
            glicko_opp_prior = glicko_dict[f"tiebreak_{surface}"][opponent]
            player_tiebreaks_won = int(match_stats_atp.loc[i,"tiebreaks_won"])
            opponent_tiebreaks_won = int(match_stats_atp.loc[i,"tiebreaks_lost"])


            for twp in range(player_tiebreaks_won):
                
                glicko_dict[f"tiebreak_{surface}"][player] = glicko_dict["envs"][f"tiebreak_{surface}"].rate(glicko_player_prior, [(WIN, glicko_opp_prior )])
                glicko_dict[f"tiebreak_{surface}"][opponent] = glicko_dict["envs"][f"tiebreak_{surface}"].rate(glicko_opp_prior, [(LOSS, glicko_player_prior )])

            for two in range(opponent_tiebreaks_won):
                
                glicko_dict[f"tiebreak_{surface}"][player] = glicko_dict["envs"][f"tiebreak_{surface}"].rate(glicko_player_prior, [(LOSS, glicko_opp_prior )])
                glicko_dict[f"tiebreak_{surface}"][opponent] = glicko_dict["envs"][f"tiebreak_{surface}"].rate(glicko_opp_prior, [(WIN, glicko_player_prior )])
                
            atp_glicko_output_dict["category"].append(f"tiebreak_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"tiebreak_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"tiebreak_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"tiebreak_{surface}"][player].sigma)

            atp_glicko_output_dict["category"].append(f"tiebreak_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"tiebreak_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"tiebreak_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"tiebreak_{surface}"][opponent].sigma)
                
        
        #set_wins
        glicko_player_prior = glicko_dict["set_wins_overall"][player]
        glicko_opp_prior = glicko_dict["set_wins_overall"][opponent]
        for set_num in range(1,6):

            #did player win the set
            set_won_flag = match_stats_atp.loc[i,f"set{set_num}_win_flag"]
            if pd.isnull(set_won_flag):
                continue
            #update player glicko using priors
            glicko_dict["set_wins_overall"][player] = glicko_dict["envs"]["set_wins_overall"].rate(glicko_player_prior, [(set_won_flag,glicko_opp_prior)])
            #update opponent glicko using priors and opposite of set won
            glicko_dict["set_wins_overall"][opponent] = glicko_dict["envs"]["set_wins_overall"].rate(glicko_opp_prior, [(int(not set_won_flag),glicko_player_prior)])
        
        atp_glicko_output_dict["category"].append("set_wins_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["set_wins_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["set_wins_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["set_wins_overall"][player].sigma)

        atp_glicko_output_dict["category"].append("set_wins_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["set_wins_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["set_wins_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["set_wins_overall"][opponent].sigma)  

        #set_wins surface
        if surface in ["clay","hard","grass","carpet"]:
            glicko_player_prior = glicko_dict[f"set_wins_{surface}"][player]
            glicko_opp_prior = glicko_dict[f"set_wins_{surface}"][opponent]
            for set_num in range(1,6):

                #did player win the set
                set_won_flag = match_stats_atp.loc[i,f"set{set_num}_win_flag"]
                if pd.isnull(set_won_flag):
                    continue
                #update player glicko using priors
                glicko_dict[f"set_wins_{surface}"][player] = glicko_dict["envs"][f"set_wins_{surface}"].rate(glicko_player_prior, [(set_won_flag,glicko_opp_prior)])
                #update opponent glicko using priors and opposite of set won
                glicko_dict[f"set_wins_{surface}"][opponent] = glicko_dict["envs"][f"set_wins_{surface}"].rate(glicko_opp_prior, [(int(not set_won_flag),glicko_player_prior)])
            
            atp_glicko_output_dict["category"].append(f"set_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"set_wins_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"set_wins_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"set_wins_{surface}"][player].sigma)

            atp_glicko_output_dict["category"].append(f"set_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"set_wins_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"set_wins_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"set_wins_{surface}"][opponent].sigma)  
        
        #service_hold_wins v return_break_wins
            
        #get latest holds glicko from player and return scores from opp and vice versa
        player_service_holds_win_glicko_prior = glicko_dict["service_hold_wins_overall"][player]
        player_return_break_wins_glicko_prior = glicko_dict["return_break_wins_overall"][player]
        opponent_service_holds_win_glicko_prior = glicko_dict["service_hold_wins_overall"][opponent]
        opponent_return_break_wins_glicko_prior = glicko_dict["return_break_wins_overall"][opponent]
        

        #init match results
        if pd.isnull(match_stats_atp.loc[i,"service_games"]) or pd.isnull(match_stats_atp.loc[i+1,"service_games"]) or match_stats_atp.loc[i,"service_games"] == 0 or match_stats_atp.loc[i+1,"service_games"] == 0:
            continue

        service_holds_player = int(match_stats_atp.loc[i,"service_games_held"])
        service_games_player = int(match_stats_atp.loc[i,"service_games"])
        service_holds_opponent = int(match_stats_atp.loc[i+1,"service_games_held"])
        service_games_opponent = int(match_stats_atp.loc[i+1,"service_games"])

        service_hold_perc_player = float(service_holds_player)/(service_games_player)
        service_hold_perc_opponent = float(service_holds_opponent)/(service_games_opponent)
        return_break_perc_player = float(service_games_opponent-service_holds_opponent)/(service_games_opponent)
        return_break_perc_opponent = float(service_games_player-service_holds_player)/(service_games_player)

        
        #rate service holds for player as wins for service_hold_wins glicko against opponent's prior return strength glicko
        glicko_dict["service_hold_wins_overall"][player] = glicko_dict["envs"]["service_hold_wins_overall"].rate(player_service_holds_win_glicko_prior, [(service_hold_perc_player,opponent_return_break_wins_glicko_prior)])
        #rate service holds for player as losses for opponent's return strength glicko against player's service hold win strength glicko
        glicko_dict["return_break_wins_overall"][opponent] = glicko_dict["envs"]["return_break_wins_overall"].rate(opponent_return_break_wins_glicko_prior, [(return_break_perc_opponent,player_service_holds_win_glicko_prior)])

        
        #rate service holds for opponent as wins for service_hold_wins glicko against player's prior return strength glicko
        glicko_dict["service_hold_wins_overall"][opponent] = glicko_dict["envs"]["service_hold_wins_overall"].rate(opponent_service_holds_win_glicko_prior, [(service_hold_perc_opponent,player_return_break_wins_glicko_prior)])
        #rate service holds for opponent as losses for opponent's return strength glicko against player's service hold win strength glicko
        glicko_dict["return_break_wins_overall"][player] = glicko_dict["envs"]["return_break_wins_overall"].rate(player_return_break_wins_glicko_prior, [(return_break_perc_player,opponent_service_holds_win_glicko_prior)])

        atp_glicko_output_dict["category"].append("service_hold_wins_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["service_hold_wins_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["service_hold_wins_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["service_hold_wins_overall"][player].sigma)

        atp_glicko_output_dict["category"].append("service_hold_wins_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["service_hold_wins_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["service_hold_wins_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["service_hold_wins_overall"][opponent].sigma)

        atp_glicko_output_dict["category"].append("return_break_wins_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["return_break_wins_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["return_break_wins_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["return_break_wins_overall"][player].sigma)

        atp_glicko_output_dict["category"].append("return_break_wins_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["return_break_wins_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["return_break_wins_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["return_break_wins_overall"][opponent].sigma)

        #serve holds and return breaks by surface
        if surface in ["hard","clay","grass","carpet"]:

            #get latest holds glicko from player and return scores from opp and vice versa
            player_service_holds_win_glicko_prior = glicko_dict[f"service_hold_wins_{surface}"][player]
            player_return_break_wins_glicko_prior = glicko_dict[f"return_break_wins_{surface}"][player]
            opponent_service_holds_win_glicko_prior = glicko_dict[f"service_hold_wins_{surface}"][opponent]
            opponent_return_break_wins_glicko_prior = glicko_dict[f"return_break_wins_{surface}"][opponent]

            

            #init match results
            if pd.isnull(match_stats_atp.loc[i,"service_games"]) or pd.isnull(match_stats_atp.loc[i+1,"service_games"]) or match_stats_atp.loc[i,"service_games"] == 0 or match_stats_atp.loc[i+1,"service_games"] == 0:
                continue

            service_holds_player = int(match_stats_atp.loc[i,"service_games_held"])
            service_games_player = int(match_stats_atp.loc[i,"service_games"])
            service_holds_opponent = int(match_stats_atp.loc[i+1,"service_games_held"])
            service_games_opponent = int(match_stats_atp.loc[i+1,"service_games"])

            service_hold_perc_player = float(service_holds_player)/(service_games_player)
            service_hold_perc_opponent = float(service_holds_opponent)/(service_games_opponent)
            return_break_perc_player = float(service_games_opponent-service_holds_opponent)/(service_games_opponent)
            return_break_perc_opponent = float(service_games_player-service_holds_player)/(service_games_player)

            
            #rate service holds for player as wins for service_hold_wins glicko against opponent's prior return strength glicko
            glicko_dict[f"service_hold_wins_{surface}"][player] = glicko_dict["envs"][f"service_hold_wins_{surface}"].rate(player_service_holds_win_glicko_prior, [(service_hold_perc_player,opponent_return_break_wins_glicko_prior)])
            #rate service holds for player as losses for opponent's return strength glicko against player's service hold win strength glicko
            glicko_dict[f"return_break_wins_{surface}"][opponent] = glicko_dict["envs"][f"return_break_wins_{surface}"].rate(opponent_return_break_wins_glicko_prior, [(return_break_perc_opponent,player_service_holds_win_glicko_prior)])

            
            #rate service holds for opponent as wins for service_hold_wins glicko against player's prior return strength glicko
            glicko_dict[f"service_hold_wins_{surface}"][opponent] = glicko_dict["envs"][f"service_hold_wins_{surface}"].rate(opponent_service_holds_win_glicko_prior, [(service_hold_perc_opponent,player_return_break_wins_glicko_prior)])
            #rate service holds for opponent as losses for opponent's return strength glicko against player's service hold win strength glicko
            glicko_dict[f"return_break_wins_{surface}"][player] = glicko_dict["envs"][f"return_break_wins_{surface}"].rate(player_return_break_wins_glicko_prior, [(return_break_perc_player,opponent_service_holds_win_glicko_prior)])

            atp_glicko_output_dict["category"].append(f"service_hold_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"service_hold_wins_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"service_hold_wins_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"service_hold_wins_{surface}"][player].sigma)

            atp_glicko_output_dict["category"].append(f"service_hold_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"service_hold_wins_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"service_hold_wins_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"service_hold_wins_{surface}"][opponent].sigma)

            atp_glicko_output_dict["category"].append(f"return_break_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"return_break_wins_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"return_break_wins_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"return_break_wins_{surface}"][player].sigma)

            atp_glicko_output_dict["category"].append(f"return_break_wins_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"return_break_wins_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"return_break_wins_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"return_break_wins_{surface}"][opponent].sigma)

        #second_serve_win
        #second_serve_return_win
        
        #get latest holds glicko from player and return scores from opp and vice versa
        player_second_serve_win_glicko_prior = glicko_dict["second_serve_win_overall"][player]
        player_second_serve_return_win_glicko_prior = glicko_dict["second_serve_return_win_overall"][player]
        opponent_second_serve_win_glicko_prior = glicko_dict["second_serve_win_overall"][opponent]
        opponent_second_serve_return_win_glicko_prior = glicko_dict["second_serve_return_win_overall"][opponent]

        #init match results
        if pd.isnull(match_stats_atp.loc[i,"second_serve_successful"]) or pd.isnull(match_stats_atp.loc[i+1,"second_serve_successful"]) or match_stats_atp.loc[i,"second_serve_successful"] == 0 or match_stats_atp.loc[i+1,"second_serve_successful"] == 0:
            continue

        #print(player)
        #print(opponent)
        #print("player_second_serve_win_glicko_prior: " + str(player_second_serve_win_glicko_prior))
        #print("player_second_serve_return_win_glicko_prior: " + str(player_second_serve_return_win_glicko_prior))
        #print("opponent_second_serve_win_glicko_prior: " + str(opponent_second_serve_win_glicko_prior))
        #print("opponent_second_serve_return_win_glicko_prior: " + str(opponent_second_serve_return_win_glicko_prior))

        second_serve_points_player = int(match_stats_atp.loc[i,"second_serve_successful"])
        second_serve_points_won_player = int(match_stats_atp.loc[i,"second_serve_points_won"])
        second_serve_points_opponent = int(match_stats_atp.loc[i+1,"second_serve_successful"])
        second_serve_points_won_opponent = int(match_stats_atp.loc[i+1,"second_serve_points_won"])

        second_serve_win_perc_player = float(second_serve_points_won_player)/(second_serve_points_player)
        second_serve_win_perc_opponent = float(second_serve_points_won_opponent)/(second_serve_points_opponent)
        second_serve_return_win_perc_player = float(second_serve_points_opponent-second_serve_points_won_opponent)/(second_serve_points_opponent)
        second_serve_return_win_perc_opponent = float(second_serve_points_player-second_serve_points_won_player)/(second_serve_points_player)


        second_serve_win_cdf_player = match_stats_atp.loc[i,"second_serve_win_perc_cdf"]
        second_serve_win_cdf_opponent = match_stats_atp.loc[i+1,"second_serve_win_perc_cdf"]
        second_serve_return_win_cdf_player = match_stats_atp.loc[i,"second_serve_return_win_perc_cdf"]
        second_serve_return_win_cdf_opponent = match_stats_atp.loc[i+1,"second_serve_return_win_perc_cdf"]

        #print("second_serve_win_cdf_player: " + str(second_serve_win_cdf_player))
        #print("second_serve_win_cdf_opponent: " + str(second_serve_win_cdf_opponent))
        #print("second_serve_return_win_cdf_player: " + str(second_serve_return_win_cdf_player))
        #print("second_serve_return_win_cdf_opponent: " + str(second_serve_return_win_cdf_opponent))

        
        #rate second serve wins for player as wins for second_serve_win glicko against opponent's prior second_serve_return_win glicko
        glicko_dict["second_serve_win_overall"][player] = glicko_dict["envs"]["second_serve_win_overall"].rate(player_second_serve_win_glicko_prior, [(second_serve_win_cdf_player,opponent_second_serve_return_win_glicko_prior)])
        #rate second serve wins for player as losses for opponent's second serve return win glicko against player's second serve win glicko
        glicko_dict["second_serve_return_win_overall"][opponent] = glicko_dict["envs"]["second_serve_return_win_overall"].rate(opponent_second_serve_return_win_glicko_prior, [(second_serve_return_win_cdf_opponent,player_second_serve_win_glicko_prior)])

        
        #rate service holds for opponent as wins for second serve win glicko against player's prior second serve return return glicko
        glicko_dict["second_serve_win_overall"][opponent] = glicko_dict["envs"]["second_serve_win_overall"].rate(opponent_second_serve_win_glicko_prior, [(second_serve_win_cdf_opponent,player_second_serve_return_win_glicko_prior)])
        #rate service holds for opponent as losses for opponent's return strength glicko against player's service hold win strength glicko
        glicko_dict["second_serve_return_win_overall"][player] = glicko_dict["envs"]["second_serve_return_win_overall"].rate(player_second_serve_return_win_glicko_prior, [(second_serve_return_win_cdf_player,opponent_second_serve_win_glicko_prior)])

        atp_glicko_output_dict["category"].append("second_serve_win_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["second_serve_win_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["second_serve_win_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["second_serve_win_overall"][player].sigma)

        atp_glicko_output_dict["category"].append("second_serve_win_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["second_serve_win_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["second_serve_win_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["second_serve_win_overall"][opponent].sigma)

        atp_glicko_output_dict["category"].append("second_serve_return_win_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(player)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["second_serve_return_win_overall"][player].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["second_serve_return_win_overall"][player].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["second_serve_return_win_overall"][player].sigma)

        atp_glicko_output_dict["category"].append("second_serve_return_win_overall")
        atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
        atp_glicko_output_dict['event_id'].append(event_id)
        atp_glicko_output_dict["player_name"].append(opponent)
        atp_glicko_output_dict["glicko2_mu"].append(glicko_dict["second_serve_return_win_overall"][opponent].mu)
        atp_glicko_output_dict["glicko2_phi"].append(glicko_dict["second_serve_return_win_overall"][opponent].phi)
        atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict["second_serve_return_win_overall"][opponent].sigma)


        #print('glicko_dict["second_serve_win_overall"][player].mu' + str(glicko_dict["second_serve_win_overall"][player].mu))
        #print('glicko_dict["second_serve_win_overall"][opponent].mu' + str(glicko_dict["second_serve_win_overall"][opponent].mu))
        #print('glicko_dict["second_serve_return_win_overall"][player].mu' + str(glicko_dict["second_serve_return_win_overall"][player].mu))
        #print('glicko_dict["second_serve_return_win_overall"][opponent].mu' + str(glicko_dict["second_serve_return_win_overall"][opponent].mu))

        if surface in ['clay','hard','grass','carpet']:
            #get latest holds glicko from player and return scores from opp and vice versa
            player_second_serve_win_glicko_prior = glicko_dict[f"second_serve_win_{surface}"][player]
            player_second_serve_return_win_glicko_prior = glicko_dict[f"second_serve_return_win_{surface}"][player]
            opponent_second_serve_win_glicko_prior = glicko_dict[f"second_serve_win_{surface}"][opponent]
            opponent_second_serve_return_win_glicko_prior = glicko_dict[f"second_serve_return_win_{surface}"][opponent]

            #init match results
            if pd.isnull(match_stats_atp.loc[i,"second_serve_successful"]) or pd.isnull(match_stats_atp.loc[i+1,"second_serve_successful"]) or match_stats_atp.loc[i,"second_serve_successful"] == 0 or match_stats_atp.loc[i+1,"second_serve_successful"] == 0:
                continue

            second_serve_points_player = int(match_stats_atp.loc[i,"second_serve_successful"])
            second_serve_points_won_player = int(match_stats_atp.loc[i,"second_serve_points_won"])
            second_serve_points_opponent = int(match_stats_atp.loc[i+1,"second_serve_successful"])
            second_serve_points_won_opponent = int(match_stats_atp.loc[i+1,"second_serve_points_won"])

            second_serve_win_perc_player = float(second_serve_points_won_player)/(second_serve_points_player)
            second_serve_win_perc_opponent = float(second_serve_points_won_opponent)/(second_serve_points_opponent)
            second_serve_return_win_perc_player = float(second_serve_points_opponent-second_serve_points_won_opponent)/(second_serve_points_opponent)
            second_serve_return_win_perc_opponent = float(second_serve_points_player-second_serve_points_won_player)/(second_serve_points_player)
            
            #rate second serve wins for player as wins for second_serve_win glicko against opponent's prior second_serve_return_win glicko
            glicko_dict[f"second_serve_win_{surface}"][player] = glicko_dict["envs"][f"second_serve_win_{surface}"].rate(player_second_serve_win_glicko_prior, [(second_serve_win_perc_player,opponent_second_serve_return_win_glicko_prior)])
            #rate second serve wins for player as losses for opponent's second serve return win glicko against player's second serve win glicko
            glicko_dict[f"second_serve_return_win_{surface}"][opponent] = glicko_dict["envs"][f"second_serve_return_win_{surface}"].rate(opponent_second_serve_return_win_glicko_prior, [(second_serve_return_win_perc_opponent,player_second_serve_win_glicko_prior)])

            
            #rate second serve wins for opponent as wins for second serve win glicko against player's prior second serve return return glicko
            glicko_dict[f"second_serve_win_{surface}"][opponent] = glicko_dict["envs"][f"second_serve_win_{surface}"].rate(opponent_second_serve_win_glicko_prior, [(second_serve_win_perc_opponent,player_second_serve_return_win_glicko_prior)])
            #rate second serve wins for opponent as losses for player's return strength glicko against player's service hold win strength glicko
            glicko_dict[f"second_serve_return_win_{surface}"][player] = glicko_dict["envs"][f"second_serve_return_win_{surface}"].rate(player_second_serve_return_win_glicko_prior, [(second_serve_return_win_perc_player,opponent_second_serve_win_glicko_prior)])

            atp_glicko_output_dict["category"].append(f"second_serve_win_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"second_serve_win_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"second_serve_win_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"second_serve_win_{surface}"][player].sigma)

            atp_glicko_output_dict["category"].append(f"second_serve_win_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"second_serve_win_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"second_serve_win_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"second_serve_win_{surface}"][opponent].sigma)

            atp_glicko_output_dict["category"].append(f"second_serve_return_win_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(player)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"second_serve_return_win_{surface}"][player].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"second_serve_return_win_{surface}"][player].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"second_serve_return_win_{surface}"][player].sigma)

            atp_glicko_output_dict["category"].append(f"second_serve_return_win_{surface}")
            atp_glicko_output_dict["date"].append(match_stats_atp.loc[i,"match_date"])
            atp_glicko_output_dict['event_id'].append(event_id)
            atp_glicko_output_dict["player_name"].append(opponent)
            atp_glicko_output_dict["glicko2_mu"].append(glicko_dict[f"second_serve_return_win_{surface}"][opponent].mu)
            atp_glicko_output_dict["glicko2_phi"].append(glicko_dict[f"second_serve_return_win_{surface}"][opponent].phi)
            atp_glicko_output_dict["glicko2_sigma"].append(glicko_dict[f"second_serve_return_win_{surface}"][opponent].sigma)

    
    


atp_glicko_output_df = pd.DataFrame(atp_glicko_output_dict)
atp_glicko_output_df = atp_glicko_output_df.sort_values(["player_name","category","date"])
atp_glicko_output_df["glicko2_mu_lower"] = atp_glicko_output_df["glicko2_mu"] - 1.96*atp_glicko_output_df["glicko2_phi"]
atp_glicko_output_df["glicko2_mu_upper"] = atp_glicko_output_df["glicko2_mu"] + 1.96*atp_glicko_output_df["glicko2_phi"]


atp_glicko_output_df["surface"] = atp_glicko_output_df["category"].str.split("_").str[-1]
atp_glicko_output_df["glicko2_category"] =  atp_glicko_output_df["category"].str.replace("_"," ")
for surface in atp_glicko_output_df["surface"].unique():
    atp_glicko_output_df["glicko2_category"] =  atp_glicko_output_df["glicko2_category"].str.replace(" "+surface,"")

atp_glicko_output_df.to_csv("../data/prep/atp_glicko_output_df.csv")

100%|██████████| 20362/20362 [00:34<00:00, 581.96it/s]


In [532]:
#ml input reshaping
#todo parameterize this in ML feature selection
glicko2_params = ["glicko2_mu", "glicko2_mu_lower"]
#this line of code ensures we only use historical data for current match glicko ratings by shifting
atp_glicko_output_df[glicko2_params] = atp_glicko_output_df.groupby(["player_name","category"])[glicko2_params].shift(1)

In [533]:
#pivot wider and rename columns
glicko_categories = atp_glicko_output_df.glicko2_category.str.replace(" ","_").unique()
glicko_pivot = atp_glicko_output_df.pivot(index=['player_name','date','event_id','surface'], columns = 'category', values = glicko2_params).reset_index()
glicko_pivot.rename({"date":"match_date"},axis=1,inplace=True)
glicko_pivot.columns = ["_".join(map(str,col)) if col[1] != '' else col[0] for col in glicko_pivot.columns.values]
glicko_pivot = glicko_pivot.drop("surface",axis=1).groupby(["player_name","match_date","event_id"]).max().reset_index().merge(match_stats[["player_name","match_date","event_id","surface"]],how="left")

In [534]:
surfaces = glicko_pivot.surface.unique()
#condense surface columns to relevant surface
for c in tqdm(glicko_categories):
    for p in glicko2_params:
        score_cols = [p + "_" + c + "_" + s for s in surfaces]

        # map each surface value to an integer column index
        col_idx = pd.Series(range(len(surfaces)), index=surfaces)

        i = glicko_pivot["surface"].map(col_idx).to_numpy()
        scores = glicko_pivot[score_cols].to_numpy()

        glicko_pivot[p + "_" + c + "_" + "surface"] = scores[np.arange(len(glicko_pivot)), i]

        #drop the redundant "_{surface}" columns
        for s in surfaces:
            glicko_pivot.drop(p + "_" + c + "_" + s, inplace=True,axis=1)

100%|██████████| 7/7 [00:00<00:00, 21.69it/s]


In [ ]:
glicko_pivot.columns = ["player_"+c if "glicko" in c else c for c in glicko_pivot.columns]
match_stats_atp_glicko = match_stats_atp.merge(glicko_pivot, how="left", on=["player_name","match_date","event_id"])
glicko_pivot.columns = glicko_pivot.columns.str.replace("player_","opponent_")
match_stats_atp_glicko = match_stats_atp_glicko.merge(glicko_pivot, how="left", on=["opponent_name","match_date","event_id"])


In [537]:
match_stats_atp_glicko

,index,match_date,event_id,season_id,competition_name,competition_category,competition_level,venue_ids,venue_capacitys,player_id,...,opponent_glicko2_mu_second_serve_return_win_surface,opponent_glicko2_mu_lower_second_serve_return_win_surface,opponent_glicko2_mu_second_serve_win_surface,opponent_glicko2_mu_lower_second_serve_win_surface,opponent_glicko2_mu_service_hold_wins_surface,opponent_glicko2_mu_lower_service_hold_wins_surface,opponent_glicko2_mu_set_wins_surface,opponent_glicko2_mu_lower_set_wins_surface,opponent_glicko2_mu_tiebreak_surface,opponent_glicko2_mu_lower_tiebreak_surface
0,0,2019-01-01,2019-0451270,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2019-01-01,2019-0451270,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2019-01-01,2019-0451271,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2019-01-01,2019-0451271,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2019-01-01,2019-0451272,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40719,132339,2026-02-08,sr:sport_event:68747386,sr:season:133607,"ATP Rotterdam, Netherlands Men Singles",ATP,atp_500,sr:venue:3696,0.0,sr:competitor:58221,...,1511.547718,1388.602818,1499.599677,1376.793458,1554.001798,1463.554636,1553.083269,1422.095761,1456.963529,1258.480094
40720,132368,2026-02-08,sr:sport_event:68748392,sr:season:133607,"ATP Rotterdam, Netherlands Men Singles",ATP,atp_500,sr:venue:1648,0.0,sr:competitor:601146,...,1456.554060,1101.270693,1475.512905,1121.790302,1549.757504,1183.609001,1495.005552,1102.703720,1273.554204,816.155439
40721,132369,2026-02-08,sr:sport_event:68748392,sr:season:133607,"ATP Rotterdam, Netherlands Men Singles",ATP,atp_500,sr:venue:1648,0.0,sr:competitor:266711,...,1499.177244,1371.400378,1570.654556,1441.879856,1735.005686,1590.953517,1717.334252,1583.998810,1632.676008,1448.073340
40722,132392,2026-02-08,sr:sport_event:68748562,sr:season:133607,"ATP Rotterdam, Netherlands Men Singles",ATP,atp_500,sr:venue:3696,0.0,sr:competitor:51141,...,1507.124895,1098.433242,1463.124977,1055.825742,1473.888319,1053.943129,1511.701383,1092.433225,1500.000000,814.000000


In [ ]:
ml_pre_input = match_stats_atp_glicko.reset_index()
ml_input_dict = pd.DataFrame()
for i in tqdm(match_stats_atp_glicko[::2].index):

    #randomize order of data 
    if random.random() < 0.5:
        player_row = ml_pre_input.loc[i,:]
        opp_row = ml_pre_input.loc[i+1,:]
    else:
        player_row = ml_pre_input.loc[i+1,:]
        opp_row = ml_pre_input.loc[i,:]

    #select glicko params and different corresponding params
    player_glicko = 
    

,index,match_date,event_id,season_id,competition_name,competition_category,competition_level,venue_ids,venue_capacitys,player_id,...,opponent_opponent_glicko2_mu_second_serve_return_win_surface,opponent_opponent_glicko2_mu_lower_second_serve_return_win_surface,opponent_opponent_glicko2_mu_second_serve_win_surface,opponent_opponent_glicko2_mu_lower_second_serve_win_surface,opponent_opponent_glicko2_mu_service_hold_wins_surface,opponent_opponent_glicko2_mu_lower_service_hold_wins_surface,opponent_opponent_glicko2_mu_set_wins_surface,opponent_opponent_glicko2_mu_lower_set_wins_surface,opponent_opponent_glicko2_mu_tiebreak_surface,opponent_opponent_glicko2_mu_lower_tiebreak_surface
0,0,2019-01-01,2019-0451270,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2019-01-01,2019-0451270,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2019-01-01,2019-0451271,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2019-01-01,2019-0451271,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2019-01-01,2019-0451272,NaN,Doha,ATP,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40709,132125,2026-02-07,sr:sport_event:68730300,sr:season:133639,"ATP Buenos Aires, Argentina Men Singles",ATP,atp_250,sr:venue:19465,0.0,sr:competitor:161262,...,1454.605568,1130.031918,1554.663114,1231.606775,1522.164475,1188.454341,1489.271379,1145.624414,1678.074535,1140.039786
40710,132136,2026-02-07,sr:sport_event:68742768,sr:season:133607,"ATP Rotterdam, Netherlands Men Singles",ATP,atp_500,sr:venue:3696,0.0,sr:competitor:54955,...,1500.805467,1088.025188,1413.672000,992.712508,1506.353719,1080.600429,1340.008810,874.387356,1500.000000,814.000000
40711,132137,2026-02-07,sr:sport_event:68742768,sr:season:133607,"ATP Rotterdam, Netherlands Men Singles",ATP,atp_500,sr:venue:3696,0.0,sr:competitor:1028097,...,1513.350218,1326.897084,1485.132991,1299.962621,1607.913332,1410.146927,1571.446564,1353.799116,1656.552920,1378.681263
40712,132150,2026-02-07,sr:sport_event:68747594,sr:season:133603,"ATP Dallas, USA Men Singles",ATP,atp_500,sr:venue:81849,0.0,sr:competitor:234046,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
match_stats_atp

In [ ]:
match_stats_atp

In [ ]:
atp_glicko_output_df.groupby(['player_name','date','category'])['glicko2_mu'].count().sort_values(ascending=False)